In [1]:
import os
import glob
import pandas as pd
import numpy as np
import string
import tempfile
import csv
from matplotlib import pyplot as plt

# we will use Gensim to train our word embeddings
from gensim.test.utils import datapath
from gensim import utils
import gensim.models
from gensim.models.callbacks import CallbackAny2Vec

# we will use NKTK for lemmatization and stopword removal
import nltk
nltk.download('popular')
from nltk.tokenize import word_tokenize
from nltk.tokenize import sent_tokenize
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
nltk.download('stopwords')

[nltk_data] Downloading collection 'popular'
[nltk_data]    | 
[nltk_data]    | Downloading package cmudict to
[nltk_data]    |     /home/jovyan/nltk_data...
[nltk_data]    |   Package cmudict is already up-to-date!
[nltk_data]    | Downloading package gazetteers to
[nltk_data]    |     /home/jovyan/nltk_data...
[nltk_data]    |   Package gazetteers is already up-to-date!
[nltk_data]    | Downloading package genesis to
[nltk_data]    |     /home/jovyan/nltk_data...
[nltk_data]    |   Package genesis is already up-to-date!
[nltk_data]    | Downloading package gutenberg to
[nltk_data]    |     /home/jovyan/nltk_data...
[nltk_data]    |   Package gutenberg is already up-to-date!
[nltk_data]    | Downloading package inaugural to
[nltk_data]    |     /home/jovyan/nltk_data...
[nltk_data]    |   Package inaugural is already up-to-date!
[nltk_data]    | Downloading package movie_reviews to
[nltk_data]    |     /home/jovyan/nltk_data...
[nltk_data]    |   Package movie_reviews is already up-to

True

In [3]:
def lemmatize_text(text):
    '''
    Lemmatizes the provided string and returns it.
    @param text : a string of unlemmatized text
    @return a lemmatized string
    '''
    lemmatizer = WordNetLemmatizer()
    lemmatized_words = [lemmatizer.lemmatize(word.lower()) for word in text.split()]
    return ' '.join(lemmatized_words)


def remove_stop_words(text):
    '''
    Removes stop words from the provided string and returns it.
    @param text : a string that contains stop words
    @return a string without stop words
    '''
    stop_words = set(stopwords.words('english'))
    filtered_sentence = [word for word in text.split() if word not in stop_words]
    return ' '.join(filtered_sentence)


def get_clean_odyssey_data():
    '''
    Pipeline to load Odyssey data into a dataframe, lemmatize it, remove stop words, and return as a dataframe.
    @return a dataframe containing preprocessed text
    '''
    odyssey_df = pd.read_csv('odyssey (1).csv')
    odyssey_df['content'] = odyssey_df['content'].apply(lemmatize_text)
    odyssey_df['content'] = odyssey_df['content'].apply(remove_stop_words)
    return odyssey_df

In [4]:
class callback(CallbackAny2Vec):
    '''Callback to print loss after each epoch.
       You should stop training your word embeddings when loss stops significantly improving. 
       You will probably need to test that out! 
       Training time selection is an art, not a science.    
    '''

    def __init__(self):
        self.epoch = 0
        self.last = 0
        self.first_run = True

    def on_epoch_end(self, model):
        if self.first_run:
            self.last = model.get_latest_training_loss()
            self.first_run = False
            print('Loss after epoch {}: {}'.format(self.epoch, self.last))
            self.epoch += 1
        else:
            loss = model.get_latest_training_loss()
            converted = loss - self.last
            print('Loss after epoch {}: {}'.format(self.epoch, converted))
            self.epoch += 1
            self.last = loss

def preprocessing(corpus):
    ''' Turns corpus into list of words and returns book content as a list of words
        @param corpus : a dataframe with preprocecced text
        @return : list of words
    '''
    training_data = []
    for ind,row in corpus.iterrows():
      training_data.append(row['content'].split(" "))
    return training_data

def train_skipgram(training_data):
    ''' Uses Word2Vec algorithm on a list of training_data to train a skipgram model (predict context given target word) 
        @param training_data : list of words preprocessed
        @return : trained model using skipgram 
    '''
    num_epochs=75
    model = gensim.models.Word2Vec(epochs=num_epochs, sg=1, sentences=training_data, min_count=5, compute_loss=True, callbacks=[callback()])
    model.save('models/gensim-sg-model-' + str(num_epochs))
    print("Model saved: " + 'models/gensim-sg-model-' + str(num_epochs))
    return model

def train_cbow(training_data):
    ''' Uses Word2Vec algorithm on a list of training_data to train a CBOW model (predict word given context) 
        @param training_data : list of words preprocessed
        @return : trained model using CBOW 
    '''
    num_epochs=30
    model = gensim.models.Word2Vec(epochs=num_epochs, sg=0, sentences=training_data, min_count=3, compute_loss=True, callbacks=[callback()])
    model.save('models/gensim-cbow-model-' + str(num_epochs))
    print("Model saved: " + 'models/gensim-cbow-model-' + str(num_epochs))
    return model

def get_ten_closest(model, target_word):
    ''' Uses cosine similarily  to find and print the 10 most similar words in the model to the target word
        @param model : trained model based on provided training data
        @param target_word : word to find 10 nearest neighbors to
    '''
    options_fields = ['word', 'sim01word', 'sim01score', 'sim02word', 'sim02score', 'sim03word', 'sim03score',
                      'sim04word', 'sim04score', 'sim05word', 'sim05score', 'sim06word', 'sim06score',
                      'sim07word', 'sim07score', 'sim08word', 'sim08score', 'sim09word', 'sim09score',
                      'sim10word', 'sim10score']
    topten = model.wv.most_similar(positive=[target_word], topn=10)
    row = [target_word]
    print("Top ten most similar words to ", target_word)
    i = 1
    for pair in topten:
      printout = str(i) + ". " + pair[0] + ", distance: " + str(pair[1])
      row.append(pair[0])
      row.append(pair[1])
      print(printout)
      i += 1

def get_ten_farthest(model, target_word):
    ''' Uses cosine similarily  to find and print the 10 most dissimilar words in the model to the target word
        @param model : trained model based on provided training data
        @param target_word : word to find 10 nearest neighbors to
    '''
    options_fields = ['word', 'neg01word', 'neg01score', 'neg02word', 'neg02score', 'neg03word', 'neg03score',
                      'neg04word', 'neg04score', 'neg05word', 'neg05score', 'neg06word', 'neg06score',
                      'neg07word', 'neg07score', 'neg08word', 'neg08score', 'neg09word', 'neg09score',
                      'neg10word', 'neg10score']
    topten = model.wv.most_similar(negative=[target_word], topn=10)
    row = [target_word]
    print("Top ten least similar words to ", target_word)
    i = 1
    for pair in topten:
      printout = str(i) + ". " + pair[0] + ", distance: " + str(pair[1])
      row.append(pair[0])
      row.append(pair[1])
      print(printout)
      i += 1

def return_similarity(model, word1, word2):
    ''' Uses cosine similarily  to return the similarity of two given words
        @param model : trained model based on provided training data
        @param word1 : first word
        @param word2 : second word to calculate similarity to first word
        @return : cosine similarity of the two words
    '''
    word_vectors = model.wv
    similarity = word_vectors.similarity(word1, word2)  # Replace with your target words
    print(f"Cosine similarity: {similarity:.4f}")

def cbow_pipeline():
    ''' Uses dataframe of Odyssey data to run preprocessing and form a list of words for training data
        Trains a CBOW model on the training data 
        @return : CBOW model based on training data from corpus
    '''
    training_data = preprocessing(corpus)
    cbow_model = train_cbow(training_data)
    return cbow_model

def skipgram_pipeline():
    ''' Uses dataframe of Odyssey data to run preprocessing and form a list of words for training data
        Trains a ski[gram model on the training data 
        @return : skipgram model based on training data from corpus
    '''
    training_data = preprocessing(corpus)
    sg_model = train_skipgram(training_data)
    return sg_model

def analogy(x1, x2, y1, model):
  result = model.wv.most_similar(positive=[y1, x2], negative=[x1])
  return result[0][0]

In [5]:
# Training 
corpus = get_clean_odyssey_data()
model = cbow_pipeline() 

Loss after epoch 0: 81399.0
Loss after epoch 1: 59694.28125
Loss after epoch 2: 58029.640625
Loss after epoch 3: 59347.546875
Loss after epoch 4: 56430.375
Loss after epoch 5: 55105.96875
Loss after epoch 6: 55846.59375
Loss after epoch 7: 54386.75
Loss after epoch 8: 53611.71875
Loss after epoch 9: 50971.0
Loss after epoch 10: 50188.0625
Loss after epoch 11: 51207.375
Loss after epoch 12: 48551.9375
Loss after epoch 13: 47778.5
Loss after epoch 14: 46887.5625
Loss after epoch 15: 45970.6875
Loss after epoch 16: 47336.6875
Loss after epoch 17: 46543.0625
Loss after epoch 18: 45938.1875
Loss after epoch 19: 45346.8125
Loss after epoch 20: 44502.75
Loss after epoch 21: 44133.5
Loss after epoch 22: 43586.875
Loss after epoch 23: 43237.375
Loss after epoch 24: 41408.25
Loss after epoch 25: 42868.375
Loss after epoch 26: 42277.625
Loss after epoch 27: 42216.375
Loss after epoch 28: 40323.125
Loss after epoch 29: 40455.0
Model saved: models/gensim-cbow-model-30


In [6]:
get_ten_closest(model, 'requital')

Top ten most similar words to  requital
1. avenger, distance: 0.850945234298706
2. fulfillment, distance: 0.8084666728973389
3. loudthundering, distance: 0.8072596192359924
4. subdue, distance: 0.7803729176521301
5. violent, distance: 0.7733721137046814
6. happiness, distance: 0.7668975591659546
7. bane, distance: 0.7637091875076294
8. beset, distance: 0.7576251029968262
9. insolently, distance: 0.7571636438369751
10. prosperity, distance: 0.7558808922767639


In [7]:
get_ten_closest(model, 'vengeance')

Top ten most similar words to  vengeance
1. weep, distance: 0.7454859614372253
2. violent, distance: 0.7432217597961426
3. regarding, distance: 0.7364773154258728
4. grant, distance: 0.7362990379333496
5. taken, distance: 0.7308023571968079
6. plan, distance: 0.7286229133605957
7. beset, distance: 0.7283639311790466
8. accursed, distance: 0.7267772555351257
9. favour, distance: 0.719900369644165
10. woeful, distance: 0.7146967053413391


In [22]:
x1 = "irus" 
x2 = "shame" # change this
y1 = "odysseus" # change this

print(analogy(x1, x2, y1, model))

awoke


In [13]:
return_similarity(model, "justice", "right")

Cosine similarity: 0.3199
